# Graph Co-localization Visualizations

Network-based visualization of marker co-localization changes between unstim and PHA conditions.
Moved from `PBMSC/pbmsc_exploration.ipynb` Section 5.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import scanpy as sc
import networkx as nx
import igraph as ig
import warnings
import os

warnings.filterwarnings('ignore')
sc.set_figure_params(figsize=(6, 6), frameon=False)
sns.set_theme(style='whitegrid')
%config InlineBackend.figure_format='retina'

figures_dir = '../PBMSC/figures'
os.makedirs(figures_dir, exist_ok=True)

In [ ]:
ADATA_PATH = '../PBMSC/cache/pbmsc_adata_final_annotated.h5ad'
adata = sc.read_h5ad(ADATA_PATH)
print(adata)
print('condition:', adata.obs.condition.value_counts().to_dict())

## Section 5 — Improved Network Visualization (iGraph)

Replaces `nx.spring_layout()` from DE.ipynb Task 4 with iGraph's Fruchterman-Reingold layout.

**Improvements over the original:**
- Stable, reproducible layout via iGraph FR algorithm
- Node size ∝ weighted degree centrality
- Edge color encodes gain magnitude
- Community hulls as colored backgrounds
- Shared stable layout between unstim and PHA networks

In [ ]:
from community import community_louvain
from scipy.spatial import ConvexHull
from scipy.optimize import linear_sum_assignment
from matplotlib.patches import Polygon as MplPolygon


def nx_to_igraph(G):
    """Convert a networkx graph to igraph, preserving edge weights."""
    nodes = list(G.nodes())
    node_idx = {n: i for i, n in enumerate(nodes)}
    edges = [(node_idx[u], node_idx[v]) for u, v in G.edges()]
    weights = [G[u][v].get('weight', 1.0) for u, v in G.edges()]
    g = ig.Graph(n=len(nodes), edges=edges)
    g.vs['name'] = nodes
    g.es['weight'] = weights
    return g, nodes


def compute_igraph_layout(G, layout='fr', seed=42, niter=500):
    """Compute igraph layout for a networkx graph. Returns {node: (x, y)} dict."""
    if G.number_of_nodes() == 0:
        return {}
    g, nodes = nx_to_igraph(G)
    if layout == 'kk':
        pos_ig = g.layout_kamada_kawai()
    elif layout == 'mds':
        pos_ig = g.layout_mds()
    else:  # default: Fruchterman-Reingold
        # igraph seed param expects a layout matrix, not int — use numpy seed instead
        np.random.seed(seed)
        pos_ig = g.layout_fruchterman_reingold(weights='weight', niter=niter)
    return {nodes[i]: (pos_ig[i][0], pos_ig[i][1]) for i in range(len(nodes))}


def draw_community_hull(ax, pos, nodes_in_community, color, alpha=0.12, padding=0.1):
    """Draw a convex hull behind nodes of a community."""
    pts = np.array([pos[n] for n in nodes_in_community if n in pos])
    if len(pts) < 3:
        for p in pts:
            circle = plt.Circle(p, radius=padding * 2, color=color, alpha=alpha, zorder=0)
            ax.add_patch(circle)
        return
    try:
        hull = ConvexHull(pts)
        hull_pts = pts[hull.vertices]
        centroid = hull_pts.mean(axis=0)
        padded = centroid + (hull_pts - centroid) * (1 + padding)
        patch = MplPolygon(padded, closed=True, facecolor=color, edgecolor=color,
                           alpha=alpha, zorder=0, linewidth=1.5)
        ax.add_patch(patch)
    except Exception:
        pass


def align_partitions(partition_ref, partition_query):
    """
    Remap partition_query IDs to match partition_ref community colors using
    the Hungarian algorithm (maximise node-level overlap).
    Returns a remapped copy of partition_query.
    """
    ref_ids = sorted(set(partition_ref.values()))
    qry_ids = sorted(set(partition_query.values()))
    shared = [n for n in partition_ref if n in partition_query]
    if not shared or not ref_ids or not qry_ids:
        return partition_query

    overlap = np.zeros((len(ref_ids), len(qry_ids)), dtype=int)
    for n in shared:
        r = ref_ids.index(partition_ref[n])
        q = qry_ids.index(partition_query[n])
        overlap[r, q] += 1

    row_ind, col_ind = linear_sum_assignment(-overlap)
    mapping = {qry_ids[q]: ref_ids[r] for r, q in zip(row_ind, col_ind)}
    next_id = max(ref_ids) + 1 if ref_ids else 0
    for q in qry_ids:
        if q not in mapping:
            mapping[q] = next_id
            next_id += 1
    return {n: mapping[c] for n, c in partition_query.items()}


def plot_marker_network_igraph(
    ax, G, partition, pos, title,
    edge_weight_scale=4, node_size_base=200,
    cmap_edges=plt.cm.plasma,
    abundance_dict=None,
    color_dict=None,
):
    """
    Draw a marker network using a pre-computed igraph layout.

    Parameters
    ----------
    abundance_dict : dict or None
        Maps node name → mean arcsinh expression. Node size ∝ abundance.
        Falls back to weighted degree centrality when None.
    border_color_dict : dict or None
        Maps node name → scalar in [0, 1] (e.g. PHA fraction).
        Encoded as border ring color using coolwarm colormap.
    """
    if G.number_of_nodes() == 0:
        ax.text(0.5, 0.5, 'No significant edges', ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')
        return

    local_pos = {n: pos[n] for n in G.nodes() if n in pos}
    community_ids = sorted(set(partition.values()))
    if color_dict is not None:
        cid_to_color = color_dict
    else:
        community_colors = plt.cm.Set3(np.linspace(0, 1, max(len(community_ids), 1)))
        cid_to_color = dict(zip(community_ids, community_colors))

    # Community hull backgrounds
    for cid in community_ids:
        members = [n for n, c in partition.items() if c == cid and n in local_pos]
        draw_community_hull(ax, local_pos, members, color=cid_to_color[cid][:3], padding=0.3)

    # Edges: color by weight magnitude
    edge_weights = [G[u][v].get('weight', 0.1) for u, v in G.edges()]
    if edge_weights:
        w_min, w_max = min(edge_weights), max(edge_weights)
        w_range = w_max - w_min if w_max > w_min else 1.0
        edge_colors = [cmap_edges((w - w_min) / w_range) for w in edge_weights]
        edge_widths = [1.0 + (w - w_min) / w_range * edge_weight_scale for w in edge_weights]
        nx.draw_networkx_edges(G, local_pos, ax=ax, width=edge_widths,
                               edge_color=edge_colors, alpha=0.6)

    # Node sizes: abundance-based or weighted-degree fallback
    if abundance_dict is not None:
        abn_vals = np.array([abundance_dict.get(n, 0.0) for n in G.nodes()])
        abn_min, abn_rng = abn_vals.min(), max(abn_vals.max() - abn_vals.min(), 1e-9)
        node_sizes = [
            node_size_base + ((abundance_dict.get(n, 0.0) - abn_min) / abn_rng) * node_size_base * 4
            for n in G.nodes()
        ]
    else:
        weighted_degree = dict(G.degree(weight='weight'))
        max_deg = max(weighted_degree.values()) if weighted_degree else 1.0
        node_sizes = [
            node_size_base + (weighted_degree.get(n, 0) / max_deg) * node_size_base * 4
            for n in G.nodes()
        ]

    node_colors = [cid_to_color[partition.get(n, 0)] for n in G.nodes()]

    nx.draw_networkx_nodes(G, local_pos, ax=ax, node_color=node_colors,
                           node_size=node_sizes, edgecolors='none')

    nx.draw_networkx_labels(G, local_pos, ax=ax, font_size=7, font_weight='bold')
    ax.set_title(f'{title}\n({G.number_of_nodes()} nodes, {G.number_of_edges()} edges)', fontsize=11)
    ax.axis('off')


print('Network visualization helpers loaded.')

In [ ]:
def compare_networks_igraph(
    adata_subset,
    conditions=('unstim', 'PHA'),
    coloc_key=None,
    threshold=None,
    resolution=1.0,
    layout='fr',
    fig_prefix='5_all'
):
    """
    Build and compare marker co-localization networks between two conditions.
    Uses iGraph FR layout (shared/stable), abundance-based node sizes,
    PHA-fraction border colors, Hungarian-aligned community colors,
    a delta network showing changed edges, and a community transfer heatmap.
    """
    # 1. Select colocalization data
    if not coloc_key:
        priority = ['spatial_asinh5_top500_var', 'HOTSPOT_top500_var', 'HOTSPOT']
        coloc_key = next((k for k in priority if k in adata_subset.obsm), None)
    if not coloc_key:
        print('No colocalization data found.'); return

    full_df = pd.DataFrame(adata_subset.obsm[coloc_key], index=adata_subset.obs_names)
    pair_cols = [c for c in full_df.columns if '/' in str(c)]
    all_weights = full_df[pair_cols].abs().mean()

    if threshold is None:
        threshold = max(all_weights.quantile(0.75), 0.01)
    print(f'Key: {coloc_key} | Threshold: {threshold:.4f}')

    # 2. Mean arcsinh abundance per protein per condition (for node sizing & border color)
    arc_mat = adata_subset.layers['arcsinh']
    if hasattr(arc_mat, 'toarray'):
        arc_mat = arc_mat.toarray()
    arc_df = pd.DataFrame(arc_mat, index=adata_subset.obs_names, columns=adata_subset.var_names)

    abundance_per_cond = {}
    for cond in conditions:
        cells = adata_subset.obs[adata_subset.obs['condition'] == cond].index
        abundance_per_cond[cond] = arc_df.loc[cells].mean().to_dict()

    # PHA fraction per protein: mu_PHA / (mu_PHA + mu_ref) → [0, 1]
    pha_frac_per_protein = {}
    for protein in adata_subset.var_names:
        mu_pha = abundance_per_cond[conditions[1]].get(protein, 0.0)
        mu_ref = abundance_per_cond[conditions[0]].get(protein, 0.0)
        denom = mu_pha + mu_ref
        pha_frac_per_protein[protein] = mu_pha / denom if denom > 0 else 0.5

    # 3. Build graphs per condition and compute per-condition weights
    graphs = {}
    partitions = {}
    cond_weights_map = {}

    for cond in conditions:
        cells = adata_subset.obs[adata_subset.obs['condition'] == cond].index
        sub_df = pd.DataFrame(adata_subset[cells].obsm[coloc_key], index=cells)
        cond_w = sub_df[pair_cols].abs().mean()
        cond_weights_map[cond] = cond_w

        G = nx.Graph()
        for pair, w in cond_w.items():
            if w >= threshold:
                m1, m2 = pair.split('/')
                if m1 != m2:
                    G.add_edge(m1, m2, weight=float(w))

        part = {}
        if G.number_of_nodes() > 0:
            part = community_louvain.best_partition(G, resolution=resolution, random_state=42)

        graphs[cond] = G
        partitions[cond] = part
        print(f'  {cond}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges, '
              f'{len(set(part.values()))} communities')

    # 4. Union graph → stable shared layout
    all_nodes = set().union(*(G.nodes() for G in graphs.values()))
    G_union = nx.Graph()
    G_union.add_nodes_from(all_nodes)
    for G in graphs.values():
        for u, v, d in G.edges(data=True):
            if G_union.has_edge(u, v):
                G_union[u][v]['weight'] = max(G_union[u][v]['weight'], d.get('weight', 1.0))
            else:
                G_union.add_edge(u, v, weight=d.get('weight', 1.0))

    print(f'Computing iGraph {layout.upper()} layout on union graph ({G_union.number_of_nodes()} nodes)...')
    fixed_pos = compute_igraph_layout(G_union, layout=layout, seed=42, niter=800)

    # 5. Align community IDs between conditions (Hungarian algorithm)
    if len(conditions) >= 2 and partitions.get(conditions[0]) and partitions.get(conditions[1]):
        partitions[conditions[1]] = align_partitions(
            partitions[conditions[0]], partitions[conditions[1]]
        )
        print('Community IDs aligned between conditions.')

    # 6. Side-by-side network plots
    fig, axes = plt.subplots(1, 2, figsize=(20, 9))
    for ax, cond in zip(axes, conditions):
        plot_marker_network_igraph(
            ax=ax,
            G=graphs[cond],
            partition=partitions[cond],
            pos=fixed_pos,
            title=f'{cond} Network',
            abundance_dict=abundance_per_cond.get(cond),

        )
    plt.suptitle(
        f'Marker Co-localization Networks (iGraph {layout.upper()}, shared layout)\n'
        'Node size = mean arcsinh abundance  |  Node color = community',
        fontsize=12
    )
    plt.tight_layout()
    plt.savefig(os.path.join(figures_dir, f'{fig_prefix}_networks.svg'), format='svg', bbox_inches='tight')
    plt.show()

    # 7. Delta network: edges where |PHA_weight − unstim_weight| exceeds 80th percentile
    w_ref = cond_weights_map[conditions[0]]
    w_alt = cond_weights_map[conditions[1]]
    delta_series = w_alt - w_ref
    delta_threshold = delta_series.abs().quantile(0.80)

    G_delta = nx.Graph()
    for pair, dw in delta_series.items():
        if abs(dw) > delta_threshold:
            m1, m2 = pair.split('/')
            if m1 != m2:
                G_delta.add_edge(m1, m2, weight=abs(float(dw)), direction=float(dw))

    if G_delta.number_of_nodes() > 0:
        delta_pos = compute_igraph_layout(G_delta, layout=layout, seed=42, niter=600)
        part_delta = community_louvain.best_partition(G_delta, resolution=resolution, random_state=42)
        delta_cids = sorted(set(part_delta.values()))
        delta_c_arr = plt.cm.Set3(np.linspace(0, 1, max(len(delta_cids), 1)))
        delta_cid_color = dict(zip(delta_cids, delta_c_arr))
        w_max_delta = delta_series.abs().max() + 1e-9

        fig, ax = plt.subplots(figsize=(12, 10))
        for u, v in G_delta.edges():
            dw = G_delta[u][v]['direction']
            color = '#d62728' if dw > 0 else '#1f77b4'
            width = 1.0 + abs(dw) / w_max_delta * 5
            if u in delta_pos and v in delta_pos:
                ax.plot([delta_pos[u][0], delta_pos[v][0]],
                        [delta_pos[u][1], delta_pos[v][1]],
                        color=color, linewidth=width, alpha=0.7, zorder=1)

        # Node sizes from mean of both conditions
        node_abns = {
            n: np.mean([abundance_per_cond[c].get(n, 0.0) for c in conditions])
            for n in G_delta.nodes()
        }
        abn_arr = np.array(list(node_abns.values()))
        abn_min_d = abn_arr.min()
        abn_rng_d = max(abn_arr.max() - abn_min_d, 1e-9)

        for node in G_delta.nodes():
            if node not in delta_pos:
                continue
            x, y = delta_pos[node]
            size = 80 + ((node_abns[node] - abn_min_d) / abn_rng_d) * 400
            c = delta_cid_color[part_delta[node]]
            ax.scatter(x, y, s=size, c=[c], zorder=2, edgecolors='black', linewidths=0.5)
            ax.text(x, y, node, ha='center', va='bottom', fontsize=7, fontweight='bold', zorder=3)

        from matplotlib.lines import Line2D
        ax.legend(handles=[
            Line2D([0], [0], color='#d62728', lw=2, label=f'{conditions[1]} gained'),
            Line2D([0], [0], color='#1f77b4', lw=2, label=f'{conditions[0]} gained'),
        ], loc='upper right', fontsize=10)
        ax.set_title(
            f'Delta Network: edges with |{conditions[1]} − {conditions[0]}| > {delta_threshold:.3f}\n'
            f'Red = {conditions[1]} up  |  Blue = {conditions[0]} up', fontsize=12
        )
        ax.axis('off')
        plt.tight_layout()
        plt.savefig(os.path.join(figures_dir, f'{fig_prefix}_delta.svg'), format='svg', bbox_inches='tight')
        plt.show()
    else:
        print('No significant delta edges to display.')

    # 8. Difference heatmap
    shared_markers = sorted(all_nodes)
    corr_mats = {}
    for cond in conditions:
        cells = adata_subset.obs[adata_subset.obs['condition'] == cond].index
        sub_df = pd.DataFrame(adata_subset[cells].obsm[coloc_key], index=cells)
        w = sub_df[pair_cols].abs().mean()
        mat = pd.DataFrame(0.0, index=shared_markers, columns=shared_markers)
        for pair, val in w.items():
            m1, m2 = pair.split('/')
            if m1 in shared_markers and m2 in shared_markers:
                mat.loc[m1, m2] = mat.loc[m2, m1] = val
        corr_mats[cond] = mat

    diff_mat = corr_mats[conditions[1]] - corr_mats[conditions[0]]
    if partitions.get(conditions[1]):
        diff_order = sorted(shared_markers,
                            key=lambda x: (partitions[conditions[1]].get(x, 999), x))
    else:
        diff_order = shared_markers
    diff_mat = diff_mat.loc[diff_order, diff_order]
    vmax = max(diff_mat.abs().values.max(), 1e-6)

    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(diff_mat, cmap='RdBu_r', center=0, vmin=-vmax, vmax=vmax,
                square=True, ax=ax, cbar_kws={'label': f'{conditions[1]} − {conditions[0]} colocalization'})
    ax.set_title(f'Co-localization Change: {conditions[1]} − {conditions[0]}\n(Red = gained, Blue = lost)')
    plt.tight_layout()
    plt.savefig(os.path.join(figures_dir, f'{fig_prefix}_diff_heatmap.svg'), format='svg', bbox_inches='tight')
    plt.show()

    # 9. Community transfer heatmap (crosstab unstim → PHA with Hungarian-aligned IDs)
    if partitions.get(conditions[0]) and partitions.get(conditions[1]):
        shared_nodes = [n for n in all_nodes
                        if n in partitions[conditions[0]] and n in partitions[conditions[1]]]
        if shared_nodes:
            ref_vals = [partitions[conditions[0]][n] for n in shared_nodes]
            qry_vals = [partitions[conditions[1]][n] for n in shared_nodes]
            transfer_df = pd.crosstab(
                pd.Series(ref_vals, name=f'{conditions[0]} community'),
                pd.Series(qry_vals, name=f'{conditions[1]} community')
            )
            fig, ax = plt.subplots(figsize=(max(5, len(transfer_df.columns)), max(4, len(transfer_df))))
            sns.heatmap(transfer_df, annot=True, fmt='d', cmap='Blues', ax=ax,
                        linewidths=0.5, cbar_kws={'label': 'Node count'})
            ax.set_title(f'Community Membership Transfer ({conditions[0]} → {conditions[1]})')
            plt.tight_layout()
            plt.savefig(os.path.join(figures_dir, f'{fig_prefix}_community_transfer.svg'), format='svg', bbox_inches='tight')
            plt.show()

    # 10. Community membership printout
    print(f'\n{"="*60}\nCOMMUNITY MEMBERSHIP\n{"="*60}')
    for cond in conditions:
        communities = {}
        for node, cid in partitions[cond].items():
            communities.setdefault(cid, []).append(node)
        print(f'\n--- {cond.upper()} ---')
        for cid, mems in sorted(communities.items(), key=lambda x: -len(x[1])):
            members_str = ', '.join(sorted(mems))
            print(f'  Community {cid} ({len(mems)}): {members_str}')

    return graphs, partitions, fixed_pos


print('compare_networks_igraph() defined.')


In [ ]:
# Run on all cells — spatial_asinh5_top500_var
graphs, partitions, layout_pos = compare_networks_igraph(
    adata,
    conditions=('unstim', 'PHA'),
    coloc_key='spatial_asinh5_top500_var',
    threshold=None,
    resolution=1.0,
    layout='fr',
    fig_prefix='5_all'
)


In [ ]:
# Run on CD8 subset — highlights T cell activation changes
cd8_for_net = adata[adata.obs['cell_type'] == 'CD8'].copy()

graphs_cd8, partitions_cd8, layout_cd8 = compare_networks_igraph(
    cd8_for_net,
    conditions=('unstim', 'PHA'),
    coloc_key='spatial_asinh5_top500_var',
    threshold=None,
    resolution=1.0,
    layout='fr',
    fig_prefix='5_cd8'
)


In [ ]:
# PHA-induced cluster extraction (gained co-expression only)
def extract_pha_induced_igraph(
    adata_subset,
    coloc_key='spatial_asinh5_top500_var',
    resolution=1.0,
    layout='fr',
    fig_prefix='5_pha_induced'
):
    """Extract and visualize clusters formed specifically in PHA stimulation."""
    full_df = pd.DataFrame(adata_subset.obsm[coloc_key], index=adata_subset.obs_names)
    pair_cols = [c for c in full_df.columns if '/' in str(c)]

    cond_weights = {}
    for cond in ('unstim', 'PHA'):
        cells = adata_subset.obs[adata_subset.obs['condition'] == cond].index
        if len(cells) == 0:
            continue
        sub = pd.DataFrame(adata_subset[cells].obsm[coloc_key], index=cells)
        cond_weights[cond] = sub[pair_cols].abs().mean()

    if len(cond_weights) < 2:
        print('Need both conditions.'); return

    diff_w = cond_weights['PHA'] - cond_weights['unstim']
    gain_threshold = diff_w[diff_w > 0].quantile(0.5)

    G_gained = nx.Graph()
    for pair, dw in diff_w.items():
        if dw > gain_threshold:
            m1, m2 = pair.split('/')
            if m1 != m2:
                G_gained.add_edge(m1, m2, weight=float(dw))

    if G_gained.number_of_nodes() == 0:
        print('No significant gained edges.'); return

    partition = community_louvain.best_partition(G_gained, resolution=resolution, random_state=42)
    pos = compute_igraph_layout(G_gained, layout=layout, seed=42, niter=600)

    fig, ax = plt.subplots(figsize=(12, 10))
    plot_marker_network_igraph(
        ax=ax, G=G_gained, partition=partition, pos=pos,
        title='PHA-Induced Co-localization Clusters'
    )
    plt.tight_layout()
    plt.savefig(os.path.join(figures_dir, f'{fig_prefix}.svg'), format='svg', bbox_inches='tight')
    plt.show()

    # Print clusters
    communities = {}
    for node, cid in partition.items():
        communities.setdefault(cid, []).append(node)
    print('\nPHA-INDUCED CLUSTERS:')
    for cid, mems in sorted(communities.items(), key=lambda x: -len(x[1])):
        members_str = ", ".join(sorted(mems))
        print(f"  Cluster {cid} ({len(mems)}): {members_str}")

    return communities, G_gained, partition


pha_clusters = extract_pha_induced_igraph(
    adata,
    coloc_key='spatial_asinh5_top500_var',
    layout='fr'
)


In [ ]:
# PHA-induced cluster extraction — CD8 subset
pha_clusters_cd8 = extract_pha_induced_igraph(
    cd8_for_net,
    coloc_key='spatial_asinh5_top500_var',
    layout='fr',
    fig_prefix='5_cd8_pha_induced'
)


### 5b. Node Size ∝ Protein Abundance (Experiment)

Re-plots the all-cell and CD8 networks using mean arcsinh abundance (pooled across both conditions) for node sizing, instead of weighted degree centrality.

In [ ]:
# --- 5b. Compute mean arcsinh abundance per protein (pooled across conditions) ---
arc_mat_all = adata.layers['arcsinh']
if hasattr(arc_mat_all, 'toarray'):
    arc_mat_all = arc_mat_all.toarray()
mean_abundance_all = dict(zip(adata.var_names, arc_mat_all.mean(axis=0)))

# Same for CD8 subset
arc_mat_cd8 = cd8_for_net.layers['arcsinh']
if hasattr(arc_mat_cd8, 'toarray'):
    arc_mat_cd8 = arc_mat_cd8.toarray()
mean_abundance_cd8 = dict(zip(cd8_for_net.var_names, arc_mat_cd8.mean(axis=0)))

print('Abundance dicts ready.')
print(f'All cells  — min={min(mean_abundance_all.values()):.3f}  max={max(mean_abundance_all.values()):.3f}')
print(f'CD8 subset — min={min(mean_abundance_cd8.values()):.3f}  max={max(mean_abundance_cd8.values()):.3f}')


In [ ]:
# --- 5b. All-cell networks — node size ∝ pooled mean arcsinh abundance ---
conditions_5b = ('unstim', 'PHA')
fig, axes = plt.subplots(1, 2, figsize=(20, 9))
for ax, cond in zip(axes, conditions_5b):
    plot_marker_network_igraph(
        ax=ax,
        G=graphs[cond],
        partition=partitions[cond],
        pos=layout_pos,
        title=f'{cond} Network',
        abundance_dict=mean_abundance_all,       # pooled abundance
    )
plt.suptitle(
    'All-cell networks — node size ∝ mean arcsinh abundance (pooled)\n'
    'Edge weight = mean |colocalization|',
    fontsize=12
)
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, '5b_all_abundance_networks.svg'), format='svg', bbox_inches='tight')
plt.show()


In [ ]:
# --- 5b. CD8 networks — node size ∝ pooled mean arcsinh abundance (CD8 cells) ---
fig, axes = plt.subplots(1, 2, figsize=(20, 9))
for ax, cond in zip(axes, conditions_5b):
    plot_marker_network_igraph(
        ax=ax,
        G=graphs_cd8[cond],
        partition=partitions_cd8[cond],
        pos=layout_cd8,
        title=f'CD8 {cond} Network',
        abundance_dict=mean_abundance_cd8,       # pooled CD8 abundance
    )
plt.suptitle(
    'CD8 networks — node size ∝ mean arcsinh abundance (CD8 pooled)\n'
    'Edge weight = mean |colocalization|',
    fontsize=12
)
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, '5b_cd8_abundance_networks.svg'), format='svg', bbox_inches='tight')
plt.show()


In [ ]:
# --- 5b. PHA-induced graph — node size ∝ pooled mean arcsinh abundance ---
_, G_pha_induced, partition_pha_induced = pha_clusters
pos_pha_induced = compute_igraph_layout(G_pha_induced, layout='fr', seed=42, niter=600)

fig, ax = plt.subplots(figsize=(12, 10))
plot_marker_network_igraph(
    ax=ax,
    G=G_pha_induced,
    partition=partition_pha_induced,
    pos=pos_pha_induced,
    title='PHA-Induced Co-localization Clusters',
    abundance_dict=mean_abundance_all,       # pooled abundance
)
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, '5b_pha_induced_abundance.svg'), format='svg', bbox_inches='tight')
plt.show()


In [ ]:
# --- 5b. CD8 PHA-induced graph — node size ∝ pooled mean arcsinh abundance (CD8) ---
_, G_pha_induced_cd8, partition_pha_induced_cd8 = pha_clusters_cd8
pos_pha_induced_cd8 = compute_igraph_layout(G_pha_induced_cd8, layout='fr', seed=42, niter=600)

fig, ax = plt.subplots(figsize=(12, 10))
plot_marker_network_igraph(
    ax=ax,
    G=G_pha_induced_cd8,
    partition=partition_pha_induced_cd8,
    pos=pos_pha_induced_cd8,
    title='CD8 PHA-Induced Co-localization Clusters',
    abundance_dict=mean_abundance_cd8,       # pooled CD8 abundance
)
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, '5b_cd8_pha_induced_abundance.svg'), format='svg', bbox_inches='tight')
plt.show()


### 5c. Shared Partition — Trackable Nodes Across Conditions

Same graphs as 5a but community detection runs **once on the union graph**, so every node carries the same color in both the unstim and PHA panels.  
Each condition then gets its own independent force-directed layout so the geometry reflects that condition's actual edge weights — letting you follow how protein communities re-arrange under stimulation.

In [ ]:
def compare_networks_shared_partition(
    adata_subset,
    conditions=('unstim', 'PHA'),
    coloc_key=None,
    threshold=None,
    resolution=1.0,
    layout='fr',
    fig_prefix='5c_all',
):
    """
    Like compare_networks_igraph but runs community detection ONCE on the union
    graph so node colors are identical across both condition panels and the delta
    graph.  Each condition gets its own independent FR layout.
    """
    if not coloc_key:
        priority = ['spatial_asinh5_top500_var', 'HOTSPOT_top500_var', 'HOTSPOT']
        coloc_key = next((k for k in priority if k in adata_subset.obsm), None)
    if not coloc_key:
        print('No colocalization data found.'); return

    full_df = pd.DataFrame(adata_subset.obsm[coloc_key], index=adata_subset.obs_names)
    pair_cols = [c for c in full_df.columns if '/' in str(c)]
    all_weights = full_df[pair_cols].abs().mean()
    if threshold is None:
        threshold = max(all_weights.quantile(0.75), 0.01)
    print(f'Key: {coloc_key} | Threshold: {threshold:.4f}')

    # Abundance per condition (node sizing + border color)
    arc_mat = adata_subset.layers['arcsinh']
    if hasattr(arc_mat, 'toarray'):
        arc_mat = arc_mat.toarray()
    arc_df = pd.DataFrame(arc_mat, index=adata_subset.obs_names, columns=adata_subset.var_names)
    abundance_per_cond = {}
    for cond in conditions:
        cells = adata_subset.obs[adata_subset.obs['condition'] == cond].index
        abundance_per_cond[cond] = arc_df.loc[cells].mean().to_dict()

    pha_frac_per_protein = {}
    for protein in adata_subset.var_names:
        mu_pha = abundance_per_cond[conditions[1]].get(protein, 0.0)
        mu_ref = abundance_per_cond[conditions[0]].get(protein, 0.0)
        denom = mu_pha + mu_ref
        pha_frac_per_protein[protein] = mu_pha / denom if denom > 0 else 0.5

    # Per-condition graphs
    graphs = {}
    cond_weights_map = {}
    for cond in conditions:
        cells = adata_subset.obs[adata_subset.obs['condition'] == cond].index
        sub_df = pd.DataFrame(adata_subset[cells].obsm[coloc_key], index=cells)
        cond_w = sub_df[pair_cols].abs().mean()
        cond_weights_map[cond] = cond_w
        G = nx.Graph()
        for pair, w in cond_w.items():
            if w >= threshold:
                m1, m2 = pair.split('/')
                if m1 != m2:
                    G.add_edge(m1, m2, weight=float(w))
        graphs[cond] = G
        print(f'  {cond}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

    # Union graph → single shared Louvain partition
    G_union = nx.Graph()
    for G in graphs.values():
        G_union.add_nodes_from(G.nodes())
        for u, v, d in G.edges(data=True):
            if G_union.has_edge(u, v):
                G_union[u][v]['weight'] = max(G_union[u][v]['weight'], d.get('weight', 1.0))
            else:
                G_union.add_edge(u, v, weight=d.get('weight', 1.0))

    shared_partition = community_louvain.best_partition(G_union, resolution=resolution, random_state=42)
    n_comm = len(set(shared_partition.values()))
    print(f'Shared partition: {n_comm} communities on {G_union.number_of_nodes()} nodes')

    comm_ids = sorted(set(shared_partition.values()))
    comm_colors = plt.cm.Set3(np.linspace(0, 1, max(len(comm_ids), 1)))
    cid_to_color = dict(zip(comm_ids, comm_colors))

    # Per-condition independent FR layouts
    cond_layouts = {}
    for cond in conditions:
        print(f'  Computing FR layout for {cond}...')
        cond_layouts[cond] = compute_igraph_layout(graphs[cond], layout=layout, seed=42, niter=800)

    # Side-by-side: own layout per condition, shared community colors
    fig, axes = plt.subplots(1, 2, figsize=(20, 9))
    for ax, cond in zip(axes, conditions):
        G   = graphs[cond]
        pos = cond_layouts[cond]
        part = {n: shared_partition.get(n, 0) for n in G.nodes()}
        plot_marker_network_igraph(
            ax=ax, G=G, partition=part, pos=pos,
            title=f'{cond}  (shared communities)',
            abundance_dict=abundance_per_cond.get(cond),
            color_dict=cid_to_color,
        )
    plt.suptitle(
        'Marker Networks — Shared Partition, Per-condition FR Layout\n'
        'Node color = shared community  |  Size = mean abundance',
        fontsize=12,
    )
    plt.tight_layout()
    plt.savefig(os.path.join(figures_dir, f'{fig_prefix}_shared_networks.svg'), format='svg', bbox_inches='tight')
    plt.show()

    # Delta network — use shared partition colors on delta nodes
    w_ref = cond_weights_map[conditions[0]]
    w_alt = cond_weights_map[conditions[1]]
    delta_series = w_alt - w_ref
    delta_threshold = delta_series.abs().quantile(0.80)

    G_delta = nx.Graph()
    for pair, dw in delta_series.items():
        if abs(dw) > delta_threshold:
            m1, m2 = pair.split('/')
            if m1 != m2:
                G_delta.add_edge(m1, m2, weight=abs(float(dw)), direction=float(dw))

    if G_delta.number_of_nodes() > 0:
        delta_pos  = compute_igraph_layout(G_delta, layout=layout, seed=42, niter=600)
        w_max_d    = delta_series.abs().max() + 1e-9
        node_abns  = {n: np.mean([abundance_per_cond[c].get(n, 0.0) for c in conditions])
                      for n in G_delta.nodes()}
        abn_arr    = np.array(list(node_abns.values()))
        abn_min_d  = abn_arr.min()
        abn_rng_d  = max(abn_arr.max() - abn_min_d, 1e-9)

        fig, ax = plt.subplots(figsize=(12, 10))
        for u, v in G_delta.edges():
            dw    = G_delta[u][v]['direction']
            color = '#d62728' if dw > 0 else '#1f77b4'
            width = 1.0 + abs(dw) / w_max_d * 5
            if u in delta_pos and v in delta_pos:
                ax.plot([delta_pos[u][0], delta_pos[v][0]],
                        [delta_pos[u][1], delta_pos[v][1]],
                        color=color, linewidth=width, alpha=0.7, zorder=1)

        for node in G_delta.nodes():
            if node not in delta_pos:
                continue
            x, y = delta_pos[node]
            size  = 80 + ((node_abns[node] - abn_min_d) / abn_rng_d) * 400
            c     = cid_to_color[shared_partition.get(node, 0)]
            ax.scatter(x, y, s=size, c=[c], zorder=2, edgecolors='none')
            ax.text(x, y, node, ha='center', va='bottom', fontsize=7, fontweight='bold', zorder=3)

        from matplotlib.lines import Line2D
        ax.legend(handles=[
            Line2D([0], [0], color='#d62728', lw=2, label=f'{conditions[1]} gained'),
            Line2D([0], [0], color='#1f77b4', lw=2, label=f'{conditions[0]} gained'),
        ], loc='upper right', fontsize=10)
        ax.set_title(
            f'Delta Network — shared partition colors\n'
            f'Red = {conditions[1]} up  |  Blue = {conditions[0]} up', fontsize=12,
        )
        ax.axis('off')
        plt.tight_layout()
        plt.savefig(os.path.join(figures_dir, f'{fig_prefix}_delta.svg'), format='svg', bbox_inches='tight')
        plt.show()
    else:
        print('No significant delta edges to display.')

    return graphs, shared_partition, cond_layouts


print('compare_networks_shared_partition() defined.')

In [ ]:

# --- 5c. All cells — shared partition ---
graphs_5c, partition_5c, layouts_5c = compare_networks_shared_partition(
    adata,
    conditions=('unstim', 'PHA'),
    coloc_key='spatial_asinh5_top500_var',
    threshold=None,
    resolution=1.0,
    layout='fr',
    fig_prefix='5c_all',
)


In [ ]:

# --- 5c. CD8 subset — shared partition ---
graphs_5c_cd8, partition_5c_cd8, layouts_5c_cd8 = compare_networks_shared_partition(
    cd8_for_net,
    conditions=('unstim', 'PHA'),
    coloc_key='spatial_asinh5_top500_var',
    threshold=None,
    resolution=1.0,
    layout='fr',
    fig_prefix='5c_cd8',
)


In [ ]:
# --- Debug: Abundance Delta vs Spatial Delta scatter (all pairs) ---
# For each spatial pair "M1/M2":
#   x = mean of per-protein abundance deltas: (mean_PHA(M1)-mean_unstim(M1) + mean_PHA(M2)-mean_unstim(M2)) / 2
#   y = spatial score delta: mean_PHA(pair) - mean_unstim(pair)

arc_mat = adata.layers['arcsinh']
pha_mask = adata.obs['condition'] == 'PHA'
unstim_mask = adata.obs['condition'] == 'unstim'

# Mean abundance per protein per condition
mean_pha_abn = pd.Series(arc_mat[pha_mask.values].mean(axis=0).A1 if hasattr(arc_mat, 'A1') else np.asarray(arc_mat[pha_mask.values].mean(axis=0)).ravel(),
                         index=adata.var_names)
mean_unstim_abn = pd.Series(arc_mat[unstim_mask.values].mean(axis=0).A1 if hasattr(arc_mat, 'A1') else np.asarray(arc_mat[unstim_mask.values].mean(axis=0)).ravel(),
                            index=adata.var_names)
delta_abn = mean_pha_abn - mean_unstim_abn  # per-protein delta

# Mean spatial score per pair per condition
sp_df = adata.obsm['spatial_asinh5_top500_var']
mean_pha_sp = sp_df[pha_mask.values].mean(axis=0)
mean_unstim_sp = sp_df[unstim_mask.values].mean(axis=0)
delta_sp = mean_pha_sp - mean_unstim_sp  # per-pair delta

# Build scatter data
rows = []
for pair in sp_df.columns:
    m1, m2 = pair.split('/')
    if m1 in delta_abn.index and m2 in delta_abn.index:
        x = (delta_abn[m1] + delta_abn[m2]) / 2
        y = delta_sp[pair]
        rows.append({'pair': pair, 'delta_abn_mean': x, 'delta_spatial': y})

scatter_df = pd.DataFrame(rows)

# Assign cell-type color: if both proteins in pair are canonical markers for same type, color by that type
_TYPE_MARKERS = {
    'B': {'CD19', 'CD20', 'CD22', 'CD268', 'CD269', 'CD24', 'CD72', 'CD79b', 'IgD', 'IgM'},
    'Platelets': {'CD9', 'CD41', 'CD42b', 'CD61', 'CD62P', 'CD36', 'CD63'},
    'CD4': {'CD4'},
    'CD8': {'CD8'},
    'NK': {'CD56', 'CD335', 'CD337', 'CD314', 'CD159a', 'CD94', 'CD16'},
    'Monocytes': {'CD14', 'CD206', 'CD163', 'CD64', 'CD33'},
}

def _pair_type(pair):
    m1, m2 = pair.split('/')
    for ct, markers in _TYPE_MARKERS.items():
        if m1 in markers or m2 in markers:
            return ct
    return 'Other'

scatter_df['cell_type'] = scatter_df['pair'].apply(_pair_type)
palette = {**_CELL_TYPE_PALETTE, 'Other': '#cccccc'}

fig, ax = plt.subplots(figsize=(10, 8))
for ct in ['Other'] + [k for k in _CELL_TYPE_PALETTE]:  # draw Other first (background)
    sub = scatter_df[scatter_df['cell_type'] == ct]
    if len(sub) == 0:
        continue
    ax.scatter(sub['delta_abn_mean'], sub['delta_spatial'],
               c=palette.get(ct, '#cccccc'), label=ct,
               s=20 if ct == 'Other' else 40,
               alpha=0.3 if ct == 'Other' else 0.7,
               edgecolors='none')

# Label extreme points
for _, row in scatter_df.nlargest(10, 'delta_spatial').iterrows():
    ax.annotate(_display_pair(row['pair']), (row['delta_abn_mean'], row['delta_spatial']),
                fontsize=7, alpha=0.8)
for _, row in scatter_df.nsmallest(5, 'delta_spatial').iterrows():
    ax.annotate(_display_pair(row['pair']), (row['delta_abn_mean'], row['delta_spatial']),
                fontsize=7, alpha=0.8)

ax.axhline(0, color='grey', lw=0.5, ls='--')
ax.axvline(0, color='grey', lw=0.5, ls='--')
ax.set_xlabel('Mean abundance delta (PHA − unstim)\navg of both proteins in pair')
ax.set_ylabel('Spatial colocalization delta (PHA − unstim)')
ax.set_title('Abundance vs Spatial change per protein pair (PHA − unstim)')
ax.legend(title='Cell type', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)
plt.tight_layout()
plt.show()

print(f'Total pairs plotted: {len(scatter_df)}')
# Show pairs where B/Platelet markers dominate
for ct in ['B', 'Platelets']:
    sub = scatter_df[scatter_df['cell_type'] == ct].sort_values('delta_spatial', ascending=False)
    print(f'\n{ct} pairs (top 5 by spatial delta):')
    print(sub[['pair', 'delta_abn_mean', 'delta_spatial']].head().to_string(index=False))
